# Tessery: portable CPU qualification on Kaggle

Use a **CPU** notebook with Internet enabled for setup. No model downloads or GPU are needed.
This runs tokenizer cancellation, queued synthetic inference, exact retrieval and SQLite
snapshot round trips. It does **not** test neural inference, Metal, GPU memory or MLX speed.
The full portable pytest suite runs first. Results include source hash and memory observations.
Save the notebook version and download `/kaggle/working/tessery-soak.json` afterwards.
A failed or interrupted run is never labelled as passed.


In [ ]:
import os
import subprocess
from pathlib import Path

root = Path("/kaggle/working/tessery")
# On a fresh notebook, fetch this branch once and record the exact revision.
# Set REF to a reviewed commit/tag when reproducing an earlier result.
REF = "main"
if not root.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/8hrsk/tessery.git", str(root)],
        check=True,
    )
subprocess.run(["git", "-C", str(root), "fetch", "--depth", "1", "origin", REF], check=True)
subprocess.run(["git", "-C", str(root), "checkout", "--detach", "FETCH_HEAD"], check=True)
revision = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
print("Tested revision:", revision)
Path("/kaggle/working/tessery-revision.txt").write_text(revision + "\n")
os.environ["METAL_INFERENCE_PORTABLE_TESTS"] = "1"
os.environ.pop("METAL_INFERENCE_TEST", None)
subprocess.run(
    [
        "python",
        "-m",
        "pip",
        "install",
        "--require-hashes",
        "--only-binary=:all:",
        "-r",
        str(root / "policy/ci-bootstrap.txt"),
    ],
    check=True,
)
subprocess.run(
    ["uv", "sync", "--locked", "--group", "dev", "--python", "3.12.13"], cwd=root, check=True
)

In [ ]:
subprocess.run(["uv", "run", "--frozen", "pytest", "-q", "-m", "not metal"], cwd=root, check=True)

The next cell explicitly starts a four-hour soak. Keep the source checkout unchanged
while it runs. The output is atomically checkpointed every 30 seconds. Traced Python
allocation growth is checked against 32 MiB; Linux current RSS is recorded separately.
This bound is not a GPU memory or process-wide leak guarantee.


In [ ]:
HOURS = 4
assert 0 < HOURS <= 24
subprocess.run(
    [
        "uv",
        "run",
        "--frozen",
        "python",
        "tools/soak_portable.py",
        "--seconds",
        str(int(HOURS * 3600)),
        "--output",
        "/kaggle/working/tessery-soak.json",
    ],
    cwd=root,
    check=True,
)

In [ ]:
import json

report = json.loads(Path("/kaggle/working/tessery-soak.json").read_text())
print({key: value for key, value in report.items() if key != "checkpoints"})
assert report["status"] == "passed"
assert report["metal_qualified"] is False